---

# 一、SWAT数据探索


## Multi-scale Residual 质量分析总结

已按照要求完成三步分析，结果如下：

**分析结果**（使用 `swat_stage1_multi_scale.pt` - 139 epochs）

### Validation Set 结果

| 指标 | 数值 | 说明 |
|------|------|------|
| Spearman ρ | 0.16 (p=0.26) | 秩相关性弱 |
| Pearson r | 0.90 (p<0.001) | 线性相关性高 |
| MAE Alignment | 0.35 | 对齐误差较小 |
| Sanity Check | 50%一致性 (5/10) | - |

### Test Set 结果

| 指标 | 数值 | 说明 |
|------|------|------|
| Spearman ρ | 0.19 (p=0.18) | 秩相关性弱 |
| Pearson r | 0.19 (p>0.05) | 线性相关性弱 |
| MAE Alignment | 3.74 | 对齐误差较大 |
| Sanity Check | 30%一致性 (3/10) | - |

### 关键发现

1. **线性 vs 秩关系不一致：**
   - Validation: Pearson 很高(0.90)，但 Spearman 很低(0.16)
   - 说明存在线性关系，但受异常值或非线性影响，秩关系不稳定

2. **Validation vs Test 差异大：**
   - Validation MAE = 0.35，Test MAE = 3.74（约 10 倍差异）
   - 说明模型在 test set 上对齐失效

3. **Scale 差异：**
   - Scale ratio ≈ 1.78x (coarse residuals 约为 fine 的 1.78 倍)
   - 数值范围差异可能导致对齐问题


## 归一化后的分析结果总结

### Validation Set（归一化后）

| 指标 | 归一化值 | 原始值 | 说明 |
|------|---------|--------|------|
| Spearman ρ | 0.1613 | 0.16 | 与原始相同，秩相关不受线性变换影响 |
| MAE Alignment | 3.7640 | 0.3484 | 归一化后变大 |
| Pearson r | 0.9028 | 0.90 | 与原始相同，线性相关不受线性变换影响 |

### Test Set（归一化后）

| 指标 | 归一化值 | 原始值 | 说明 |
|------|---------|--------|------|
| Spearman ρ | 0.1889 | 0.19 | 与原始相同 |
| MAE Alignment | 41.0673 | 3.7441 | 归一化后显著变大 |
| Pearson r | 0.1913 | 0.19 | 与原始相同 |

### 结果说明

**Stage 1 的问题不是"尺度没对齐 / 线性缩放没对齐"，而是"跨尺度 residual 的排序语义本身就没对齐"。**

我们在 validation 上调出来的阈值/决策规则，隐含假设是 val 和 test 的 score/residual 分布相近；但在 SWaT 上这个假设不成立，因此阈值迁移失败，导致 test 的 classification 指标大幅下降。


---

# 二、PSM数据子集

## Stage 1 实验结果

### 2.1 Coarse-only Baseline

重新采样后 coarse-only 的结果训练完成（50 epochs）

| 指标 | 数值 |
|------|------|
| 最佳 Val F1 | 0.5000 (epoch 43) |
| ROC-AUC | 0.7409 |
| Signal Ratio | 1.29x |
| Precision | 0.9792 |
| Recall | 0.2070 |

#### 详细评估指标

| 指标 | 数值 |
|------|------|
| Threshold | 1.208248 |
| F1 Score | 0.3247 |
| Precision | 1.0000 |
| Recall | 0.1938 |
| ROC-AUC | 0.7372 |
| AUPRC | 0.6415 |

#### Residual 分离统计

**Validation Set:**
- Normal: n=954, median=0.721877, mean=0.754700, std=0.215113
- Anomaly: n=45, median=1.226002, mean=1.465348, std=0.800965

**Test Set:**
- Normal: n=775, median=0.705602, mean=0.713330, std=0.155591
- Anomaly: n=227, median=0.970950, mean=0.942013, std=0.277248

**Signal Ratio (Anomaly/Normal median):** 1.38x  
**ROC-AUC:** 0.7372


### 2.2 Multi-scale

| 指标 | 数值 |
|------|------|
| Threshold | 1.314560 |
| F1 Score | 0.3123 |
| Precision | 1.0000 |
| Recall | 0.1850 |
| ROC-AUC | 0.7931 |
| AUPRC | 0.7078 |
| Signal Ratio | 1.50x |

#### Multi-Scale Training Summary

- ✓ Pipeline: OK (data loaded and processed)
- ✓ Split: OK (train/val/test split maintained)
- ✓ Residual Separation: GOOD (ROC-AUC = 0.7931 > 0.5)
- ✓ Signal Ratio: GOOD (1.50x > 1.0x)

**关键发现：**
- multi-scale 明确提升了 residual 的排序质量（ROC-AUC 从 0.74 → 0.79）
- 异常相对正常的偏移更大、更集中（Signal Ratio 1.29x → 1.50x）
- Precision 没塌，Recall 也没乱涨 


## Stage 2 实验结果

### 2.3 Coarse-only Baseline (Stage 2)

**配置：** COMBINED MODE (spatial_weight=0.5, temporal_weight=0.5)

**训练统计：**
- Reference computed from 26431 normal training windows
- Reference causal matrix shape: (25, 25)
- Training normal residual stats: median=0.687850, MAD=0.155101

**Score 统计（all windows）：**
- Mean: 1.302541, Std: 0.898732, Median: 1.137438
- Q90: 2.292513, Q95: 2.651477, Q99: 4.486622

**Deviation 统计：**
- Spatial: Mean=1.251269, Median=0.620954
- Temporal: Mean=1.353813, Median=1.013529

#### 评估结果

**Threshold (99% percentile of train normal):** 4.813789

**Test Set Metrics (Combined):**

| 指标 | 数值 | 状态 |
|------|------|------|
| AUPRC | 0.4141 | ✗ FAIL |
| ROC-AUC | 0.6185 | ✓ PASS |
| Normal score median | 1.006517 | - |
| Attack score median | 1.434590 | - |
| Effect size (attack/normal) | 1.43x | ✓ PASS |

**Test Set Deviation Breakdown：**
- Spatial deviation median: Normal=0.760786, Attack=0.567231
- Temporal deviation median: Normal=0.913023, Attack=1.945260

**Validation Set (reference)：**
- Normal score median: 1.026308
- Attack score median: 0.777062
- Effect size: 0.76x

**Decision Criteria：**
- AUPRC > 0.5: FAIL (0.4141)
- ROC-AUC > 0.5: PASS (0.6185)
- Effect size > 1.2x: PASS (1.43x)

✗ **Stage 2 validation FAILED - Consider fixing Stage 1**


### 2.4 Multi-scale (Stage 2)

**配置：** COMBINED MODE (spatial_weight=0.5, temporal_weight=0.5)

**训练统计：**
- Reference computed from 26431 normal training windows
- Reference causal matrix shape: (25, 25)
- Training normal residual stats: median=0.560375, MAD=0.143833

**Score 统计（all windows）：**
- Mean: 1.583050, Std: 0.963206, Median: 1.395254
- Q90: 2.417210, Q95: 3.115673, Q99: 5.797843

**Deviation 统计：**
- Spatial: Mean=1.546597, Median=1.221217
- Temporal: Mean=1.619503, Median=1.083507

#### 评估结果

**Threshold (99% percentile of train normal):** 6.109545

**Test Set Metrics (Combined):**

| 指标 | 数值 | 状态 |
|------|------|------|
| AUPRC | 0.5323 | ✓ PASS |
| ROC-AUC | 0.7003 | ✓ PASS |
| Normal score median | 1.485022 | - |
| Attack score median | 2.130195 | - |
| Effect size (attack/normal) | 1.43x | ✓ PASS |

**Test Set Deviation Breakdown：**
- Spatial deviation median: Normal=1.249314, Attack=1.102759
- Temporal deviation median: Normal=1.387190, Attack=2.856875

**Validation Set (reference)：**
- Normal score median: 1.122738
- Attack score median: 1.323391
- Effect size: 1.18x

**Decision Criteria：**
- AUPRC > 0.5: PASS (0.5323)
- ROC-AUC > 0.5: PASS (0.7003)
- Effect size > 1.2x: PASS (1.43x)

✓ **Stage 2 validation PASSED - Worth continuing!**

**关键发现：** Temporal deviation（真正的主力）所以后面可能考虑拉高比例？，目前是0.5 0.5


## PSM数据子集总结

### Stage 1：
- ✅ coarse-only baseline
- ✅ multi-scale（w/ L_u） residual 质量提升（ROC-AUC、Signal Ratio、AUPRC）

### Stage 2（Track A）：
- ✅ combined spatial+temporal 在 test 上通过（AUPRC>0.5、ROC>0.5、effect size>1.2）

### 关键诊断：
- ✅ 信号主要来自 temporal，spatial 更像 regularizer


---

# 三、PSM Full Data

## Stage 1 实验结果

### 3.1 Coarse-only Baseline

**训练命令：**
```bash
python scripts/train_stage1.py --data_dir data/PSM/processed --model_save_path models/psm_coarse_only.pt --coarse_only
```

**数据加载：**
- Coarse scale: torch.Size([44053, 60, 25])
  - Train samples: 26431
  - Val samples: 8810
  - Test samples: 8812
- Fine scale: torch.Size([220262, 60, 25])
  - Fine train/val/test: 132157/44052/44053

**标签分布：**
- Train - Normal: 26431, Anomaly: 0
- Val - Normal: 6594, Anomaly: 2216
- Test - Normal: 6101, Anomaly: 2711

#### 训练过程

**训练配置：**
- Coarse only: True
- Use LU loss: False
- Epochs: 50
- Learning rate: 0.0003
- Batch size: 32
- Device: cuda

**训练进度：**
- Epoch 10/50 - Loss: 0.427751, Val AUC-ROC: 0.5696
- Epoch 20/50 - Loss: 0.332166, Val AUC-ROC: 0.5703
- Epoch 30/50 - Loss: 0.163897, Val AUC-ROC: 0.5025
- Epoch 40/50 - Loss: 0.134179, Val AUC-ROC: 0.5079
- Epoch 50/50 - Loss: 0.105582, Val AUC-ROC: 0.5581

**训练完成：** Best AUC-ROC: 0.5725 at epoch 17. Final model saved at epoch 50.

#### 评估结果

| 指标 | 数值 |
|------|------|
| Threshold | 0.829505 |
| F1 Score | 0.5366 |
| Precision | 0.4740 |
| Recall | 0.6182 |
| ROC-AUC | 0.7154 |
| AUPRC | 0.5474 |

**Training Summary：**
- ✓ Pipeline: OK (data loaded and processed)
- ✓ Split: OK (train/val/test split maintained)
- ✓ Residual Separation: GOOD (ROC-AUC = 0.7154 > 0.5)


### 3.2 Multi-scale

#### 评估结果

| 指标 | 数值 |
|------|------|
| Threshold | 0.539715 |
| F1 Score | 0.5156 |
| Precision | 0.3519 |
| Recall | 0.9646 |
| ROC-AUC | 0.7621 |
| AUPRC | 0.6088 |

#### Residual 分离统计

**Validation Set：**
- Normal: n=6594, median=0.508040, mean=0.519661, std=0.122647
- Anomaly: n=2216, median=0.608016, mean=0.703629, std=0.457523

**Test Set：**
- Normal: n=6101, median=0.642025, mean=0.668859, std=0.156572
- Anomaly: n=2711, median=0.860232, mean=0.850616, std=0.201487

**Signal Ratio (Anomaly/Normal median):** 1.34x  
**ROC-AUC:** 0.7621

**Training Summary：**
- ✓ Pipeline: OK (data loaded and processed)
- ✓ Split: OK (train/val/test split maintained)
- ✓ Residual Separation: GOOD (ROC-AUC = 0.7621 > 0.5)


## PSM Full Data 对比分析

**说明：** 我们用的是 spatial + temporal deviation 各自0.5组成分数  
**值得注意的是实际异常比例是 30.76%**

### Top 5% Threshold Comparison

| Model | F1 | Precision | Recall | ROC-AUC | AUPRC |
|-------|----|----------|--------|---------|-------|
| psm_stage1_multiscale_full | 0.2113 | 0.7551 | 0.1228 | 0.7621 | 0.6088 |
| psm_stage1_coarse_only_fixed | 0.1986 | 0.7098 | 0.1155 | 0.6979 | 0.5283 |

### Top 10% Threshold Comparison

| Model | F1 | Precision | Recall | ROC-AUC | AUPRC |
|-------|----|----------|--------|---------|-------|
| psm_stage1_multiscale_full | 0.3640 | 0.7415 | 0.2412 | 0.7621 | 0.6088 |
| psm_stage1_coarse_only_fixed | 0.3067 | 0.6247 | 0.2032 | 0.6979 | 0.5283 |

### Top 15% Threshold Comparison

| Model | F1 | Precision | Recall | ROC-AUC | AUPRC |
|-------|----|----------|--------|---------|-------|
| psm_stage1_multiscale_full | 0.4805 | 0.7330 | 0.3574 | 0.7621 | 0.6088 |
| psm_stage1_coarse_only_fixed | 0.3809 | 0.5809 | 0.2833 | 0.6979 | 0.5283 |

### Top 20% Threshold Comparison

| Model | F1 | Precision | Recall | ROC-AUC | AUPRC |
|-------|----|----------|--------|---------|-------|
| psm_stage1_multiscale_full | 0.5512 | 0.6994 | 0.4548 | 0.7621 | 0.6088 |
| psm_stage1_coarse_only_fixed | 0.4309 | 0.5468 | 0.3556 | 0.6979 | 0.5283 |

### Top 25% Threshold Comparison

| Model | F1 | Precision | Recall | ROC-AUC | AUPRC |
|-------|----|----------|--------|---------|-------|
| psm_stage1_multiscale_full | 0.5645 | 0.6296 | 0.5116 | 0.7621 | 0.6088 |
| psm_stage1_coarse_only_fixed | 0.4807 | 0.5361 | 0.4356 | 0.6979 | 0.5283 |

### Top 30% Threshold Comparison

| Model | F1 | Precision | Recall | ROC-AUC | AUPRC |
|-------|----|----------|--------|---------|-------|
| psm_stage1_multiscale_full | 0.5651 | 0.5722 | 0.5581 | 0.7621 | 0.6088 |
| psm_stage1_coarse_only_fixed | 0.5135 | 0.5200 | 0.5072 | 0.6979 | 0.5283 |


---

# 四、SMAP数据（未补充完整）

## 训练记录

**状态：** 仅仅训练没有做evaluation  
**注意：** validation上anomaly为0
